In [ ]:
import os
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import contextily as cx
from shapely.geometry import LineString
from network_tracing import bearing_at_endpoint, deflection_deg, opposite_end, build_node_index
from config import (OUTPUTS_DIR, NODE_DECIMALS, CORRIDOR_WIDTH, CORE_DEFLECTION_THRESHOLD,
                     ARTERY_DEFLECTION_THRESHOLD, CUMULATIVE_DEFLECTION_CAP)


In [ ]:
street_gdf = gpd.read_file(os.path.join(OUTPUTS_DIR, "network_centrality.gpkg"))
street_gdf.plot()

In [ ]:
node_index = build_node_index(street_gdf, NODE_DECIMALS)

oxford_ref = LineString([
    (-0.1590, 51.5136),  # Marble Arch end
    (-0.1310, 51.5165),  # Tottenham Court Rd / Centre Point end
])
oxford_ref_gdf = gpd.GeoSeries([oxford_ref], crs="EPSG:4326").to_crs(street_gdf.crs)
corridor = oxford_ref_gdf.buffer(CORRIDOR_WIDTH).iloc[0]

def trace_core(seed_idx, direction_end, corridor_geom, max_hops=200):
    chain = [seed_idx]
    visited = {seed_idx}
    current_idx, current_exit_end = seed_idx, direction_end

    for _ in range(max_hops):
        geom = street_gdf.loc[current_idx, 'geometry']
        coords = list(geom.coords)
        exit_key = tuple(np.round(coords[-1] if current_exit_end == 'end' else coords[0], NODE_DECIMALS))
        incoming_bearing = bearing_at_endpoint(geom, which=current_exit_end)

        candidates = [(oi, w) for oi, w in node_index[exit_key] if oi != current_idx]
        if not candidates:
            break
        candidates = [(oi, w) for oi, w in candidates
                      if street_gdf.loc[oi, 'geometry'].intersects(corridor_geom)]
        if not candidates:
            break

        scored = sorted(
            ((deflection_deg(incoming_bearing + np.pi, bearing_at_endpoint(street_gdf.loc[oi, 'geometry'], w)), oi, w)
             for oi, w in candidates),
            key=lambda t: t[0]
        )
        best_defl, best_idx, best_which = scored[0]
        if best_defl > CORE_DEFLECTION_THRESHOLD or best_idx in visited:
            break

        chain.append(best_idx)
        visited.add(best_idx)
        current_idx, current_exit_end = best_idx, opposite_end(best_which)

    return chain

centroid = oxford_ref_gdf.iloc[0].centroid
candidate_dists = street_gdf.geometry.distance(centroid)
seed_idx = candidate_dists.idxmin()
print(f"Seed segment: {seed_idx}, distance to reference centroid: {candidate_dists.min():.1f}m")

oxford_idx = set(trace_core(seed_idx, 'end', corridor)) | set(trace_core(seed_idx, 'start', corridor))
oxford_street = street_gdf.loc[list(oxford_idx)]
oxford_mask = street_gdf.index.isin(oxford_idx)

print(f"Oxford Street segments (topologically traced): {len(oxford_street)}")

In [ ]:
direct_junctions = set()
for idx in oxford_idx:
    coords = list(street_gdf.loc[idx, 'geometry'].coords)
    for key in (tuple(np.round(coords[0], NODE_DECIMALS)),
                tuple(np.round(coords[-1], NODE_DECIMALS))):
        for other_idx, _ in node_index[key]:
            if other_idx not in oxford_idx:
                direct_junctions.add(other_idx)

print(f"Directly-touching segments: {len(direct_junctions)}")

In [ ]:
def trace_artery(seed_idx, seed_entry_end, max_hops=50):
    artery = [seed_idx]
    visited = {seed_idx}
    current_idx, current_exit_end = seed_idx, opposite_end(seed_entry_end)
    entry_bearing = bearing_at_endpoint(street_gdf.loc[seed_idx, 'geometry'], which=current_exit_end)

    for _ in range(max_hops):
        geom = street_gdf.loc[current_idx, 'geometry']
        coords = list(geom.coords)
        exit_key = tuple(np.round(coords[-1] if current_exit_end == 'end' else coords[0], NODE_DECIMALS))
        incoming_bearing = bearing_at_endpoint(geom, which=current_exit_end)

        candidates = [(oi, w) for oi, w in node_index[exit_key] if oi != current_idx]
        if not candidates:
            break

        scored = sorted(
            ((deflection_deg(incoming_bearing + np.pi, bearing_at_endpoint(street_gdf.loc[oi, 'geometry'], w)), oi, w)
             for oi, w in candidates),
            key=lambda t: t[0]
        )
        best_defl, best_idx, best_which = scored[0]
        if best_defl > ARTERY_DEFLECTION_THRESHOLD or best_idx in visited:
            break

        candidate_bearing = bearing_at_endpoint(street_gdf.loc[best_idx, 'geometry'], which=opposite_end(best_which))
        if deflection_deg(entry_bearing, candidate_bearing) > CUMULATIVE_DEFLECTION_CAP:
            break

        artery.append(best_idx)
        visited.add(best_idx)
        current_idx, current_exit_end = best_idx, opposite_end(best_which)

    return artery

arteries = {}
for idx in direct_junctions:
    coords = list(street_gdf.loc[idx, 'geometry'].coords)
    for which, key in (('start', tuple(np.round(coords[0], NODE_DECIMALS))),
                        ('end', tuple(np.round(coords[-1], NODE_DECIMALS)))):
        if any(oi in oxford_idx for oi, _ in node_index[key]):
            arteries[idx] = trace_artery(idx, which)
            break

In [ ]:
all_artery_idx = set(i for chain in arteries.values() for i in chain)
arteries_gdf = street_gdf.loc[list(all_artery_idx)]

print(f"Seed junction segments: {len(arteries)}")
print(f"Total segments across all arteries: {len(all_artery_idx)}")

In [ ]:
fig, ax = plt.subplots(figsize=(14, 8))

# Crossing streets drawn first, Oxford Street on top (zorder) so the
# corridor reads as one continuous line rather than being broken up
# visually by the streets crossing through it.
# crossing.to_crs(epsg=3857).plot(
#     ax=ax, color="steelblue", linewidth=1, zorder=1, label="Crossing streets"
# )
oxford_street.to_crs(epsg=3857).plot(
    ax=ax, color="red", linewidth=2.5, zorder=2, label="Oxford Street"
)

# contextily basemaps are Web Mercator (EPSG:3857) — reproject for
# plotting only, working CRS (EPSG:32630) stays untouched elsewhere.
cx.add_basemap(ax, source="https://server.arcgisonline.com/ArcGIS/rest/services/Canvas/World_Light_Gray_Base/MapServer/tile/{z}/{y}/{x}")

ax.set_axis_off()
ax.legend(loc="lower right", frameon=True)
ax.set_title("Oxford Street topologically traced and intersecting arteries")

plt.tight_layout()
plt.savefig(os.path.join(OUTPUTS_DIR, "oxford_street_crossings.png"), dpi=300)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(14, 8), dpi=300)

arteries_gdf.to_crs(epsg=3857).plot(
    ax=ax, color="orange", linewidth=1.5, zorder=1.5, label="Arteries"
)
oxford_street.to_crs(epsg=3857).plot(
    ax=ax, color="red", linewidth=2.5, zorder=2, label="Oxford Street"
)

# contextily basemaps are Web Mercator (EPSG:3857) — reproject for
# plotting only, working CRS (EPSG:32630) stays untouched elsewhere.
cx.add_basemap(ax, source="https://server.arcgisonline.com/ArcGIS/rest/services/Canvas/World_Light_Gray_Base/MapServer/tile/{z}/{y}/{x}")

ax.set_axis_off()
ax.legend(loc="upper right", frameon=True)
ax.set_title("Oxford Street and intersecting arteries")

plt.tight_layout()
plt.savefig(os.path.join(OUTPUTS_DIR, "oxford_street_crossings.png"), dpi=300)
plt.show()

In [ ]:
# import geopandas as gpd
# from shapely.geometry import LineString

# # ── 1. Define Oxford Street as a reference line ──────────────────────────
# # Rough coordinates along the corridor (Marble Arch -> Tottenham Court Rd),
# # in WGS84 lon/lat. Doesn't need to be precise — it's just a spine to buffer.
# oxford_ref = LineString([
#     (-0.1590, 51.5136),  # Marble Arch end
#     (-0.1310, 51.5165),  # Tottenham Court Rd / Centre Point end
# ])
# oxford_ref_gdf = gpd.GeoSeries([oxford_ref], crs="EPSG:4326").to_crs(street_gdf.crs)

# # ── 2. Buffer the reference line into a corridor ──────────────────────────
# # 30m either side — covers carriageway + pavement + neatnet/QGIS snapping
# # drift. Tightened up from an initial 15m, which missed segments near the
# # pedestrianised section.
# corridor = oxford_ref_gdf.buffer(30)

# # ── 3. Capture segments falling inside the corridor ───────────────────────
# oxford_mask = street_gdf.geometry.intersects(corridor.iloc[0])
# oxford_street = street_gdf[oxford_mask]

# # ── 4. Dissolve into a single line, then find what touches it ─────────────
# # A tight buffer here (not the wide corridor) — this is testing genuine
# # junction contact, not proximity. Wide would catch parallel streets that
# # never actually meet Oxford Street.
# oxford_dissolved = gpd.GeoSeries(
#     [oxford_street.geometry.union_all()], crs=street_gdf.crs
# )
# touch_buffer = oxford_dissolved.buffer(500)

# crossing = street_gdf[
#     street_gdf.geometry.intersects(touch_buffer.iloc[0]) & ~oxford_mask
# ]

# print(f"Oxford Street segments: {oxford_mask.sum()}")
# print(f"Crossing segments: {len(crossing)}")